In [ ]:
# READ-ONLY PREFLIGHT: this runs before installation, downloads, or model work.
from pathlib import Path
import shutil
import numpy as np
import pandas as pd

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
MIN_CHECKPOINT = 19_000

print("COSMIC SWAN RESUME PREFLIGHT")
print("=" * 36)

data_bundles = []
for manifest_path in INPUT.rglob("manifest.csv"):
    folder = manifest_path.parent
    features_path = folder / "features.csv"
    series_path = folder / "series.npy"
    if not (features_path.exists() and series_path.exists()):
        continue
    try:
        manifest = pd.read_csv(manifest_path, usecols=["status"])
        complete_count = int(manifest["status"].astype(str).isin(["COMPLETE", "ARCHIVED"]).sum())
        feature_count = int(len(pd.read_csv(features_path, usecols=["source_id"])))
        sequence_shape = np.load(series_path, mmap_mode="r").shape
        data_bundles.append({
            "path": folder,
            "products": complete_count,
            "features": feature_count,
            "sequences": int(sequence_shape[0]) if sequence_shape else -1,
        })
    except Exception as exc:
        print(f"CHECK: Could not read {folder}: {type(exc).__name__}: {exc}")

for bundle in sorted(data_bundles, key=lambda item: item["products"], reverse=True):
    print(
        f"Attached data: {bundle['path']} | products={bundle['products']:,} | "
        f"features={bundle['features']:,} | sequences={bundle['sequences']:,}"
    )

valid_bundles = [
    bundle for bundle in data_bundles
    if bundle["products"] >= MIN_CHECKPOINT
    and bundle["features"] >= MIN_CHECKPOINT
    and bundle["features"] == bundle["sequences"]
]
checkpoint_ok = bool(valid_bundles)
print("[" + ("PASS" if checkpoint_ok else "FAIL") + "] 19K checkpoint with matching features/sequences")

project_files = [
    p for p in INPUT.rglob("pyproject.toml")
    if (p.parent / "src/cosmic_black_swan/cli.py").exists()
]
project_ok = bool(project_files)
print("[" + ("PASS" if project_ok else "FAIL") + "] Cosmic Swan project source attached")
if project_files:
    print("Project source:", project_files[0].parent)

free_gib = shutil.disk_usage(WORK).free / (1024 ** 3)
disk_ok = free_gib >= 3.0
print(f"[{'PASS' if disk_ok else 'FAIL'}] Working space: {free_gib:.2f} GiB free (need at least 3 GiB)")

if not (checkpoint_ok and project_ok and disk_ok):
    raise RuntimeError(
        "Preflight failed. Stop here: fix the missing/wrong notebook input or free-space issue, "
        "then run again. No downloads or model stages have started."
    )

print()
print("ALL PREFLIGHT CHECKS PASSED. The following notebook cells may now start the resume run.")

In [ ]:
from pathlib import Path
import importlib.util
import json
import shutil
import subprocess
import sys
import time
import traceback
from datetime import datetime, timezone

import numpy as np
import pandas as pd

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
DATA = WORK / "cbs-data"
PROJECT_SOURCE_CANDIDATES = [
    INPUT / "notebooks/urajkaushik/cosmic-swan-4/cbs-project-src",
    INPUT / "datasets/urajkaushik/cosmic-swan-4/cbs-project-src",
]
# Search every attached notebook output, then select the bundle with the most
# COMPLETE/ARCHIVED products. This must prefer the 19K checkpoint over the 5K baseline.

PROJECT_SOURCE = next(
    (p for p in PROJECT_SOURCE_CANDIDATES if (p / "pyproject.toml").exists()), None
)
if PROJECT_SOURCE is None:
    for p in INPUT.rglob("pyproject.toml"):
        candidate = p.parent
        if (candidate / "src/cosmic_black_swan/cli.py").exists():
            PROJECT_SOURCE = candidate
            break
if PROJECT_SOURCE is None:
    raise FileNotFoundError(
        "Attach the Cosmic Swan 4 notebook output; cbs-project-src/pyproject.toml was not found."
    )

data_candidates = []
for manifest_file in INPUT.rglob("manifest.csv"):
    folder = manifest_file.parent
    if all((folder / name).exists() for name in ("features.csv", "series.npy")):
        data_candidates.append(folder)
# Deduplicate paths, then select the attached checkpoint with the largest manifest.
data_candidates = list(dict.fromkeys(data_candidates))
if not data_candidates:
    raise FileNotFoundError(
        "Attach the Cosmic Swan 4 output containing cbs-data/manifest.csv, features.csv, and series.npy."
    )

def manifest_count(folder):
    frame = pd.read_csv(folder / "manifest.csv", usecols=["status"])
    return int(frame["status"].astype(str).isin(["COMPLETE", "ARCHIVED"]).sum())

DATA_SOURCE = max(data_candidates, key=manifest_count)
checkpoint_products = manifest_count(DATA_SOURCE)
if checkpoint_products < 19_000:
    raise RuntimeError(
        f"Resume checkpoint missing or too small: found {checkpoint_products} completed products at {DATA_SOURCE}. "
        "Attach Version 2 output containing cbs-data/manifest.csv, features.csv, and series.npy; "
        "also attach Cosmic Swan 4 for the original project source."
    )
PROJECT = WORK / "cbs-project-src"
PROJECT.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(PROJECT_SOURCE, PROJECT, dirs_exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)
for name in ("manifest.csv", "features.csv", "series.npy"):
    destination = DATA / name
    if not destination.exists():
        shutil.copy2(DATA_SOURCE / name, destination)
for folder_name in ("scores", "pixel_audits"):
    source_folder = DATA_SOURCE / folder_name
    dest_folder = DATA / folder_name
    if source_folder.exists():
        shutil.copytree(source_folder, dest_folder, dirs_exist_ok=True)

# The attached raw FITS are read-only and the 5,000 rows already have matching features/sequences.
# Mark those archived in the writable manifest so featurize(append=True) processes only new files.
manifest_path = DATA / "manifest.csv"
manifest = pd.read_csv(manifest_path, dtype={"source_id": "string"})
if "local_path" in manifest:
    completed = manifest["status"].astype(str).eq("COMPLETE")
    local_paths = manifest["local_path"].fillna("").astype(str)
    missing_old = completed & ~local_paths.map(lambda value: bool(value) and Path(value).exists())
    manifest.loc[missing_old, "status"] = "ARCHIVED"
    manifest.loc[missing_old, "local_path"] = ""
    if "message" in manifest:
        manifest.loc[missing_old, "message"] = "Raw FITS retained in attached read-only input; features and series are present."
    manifest.to_csv(manifest_path, index=False)

print("Project source:", PROJECT_SOURCE)
print("Writable project copy:", PROJECT)
print("Seed data:", DATA_SOURCE)
print("Working data root:", DATA)
print("Seed completed manifest products:", manifest_count(DATA_SOURCE))
seed_features = pd.read_csv(DATA / "features.csv")
seed_sequences = np.load(DATA / "series.npy", mmap_mode="r")
if len(seed_features) < 19_000 or len(seed_features) != len(seed_sequences):
    raise RuntimeError(
        f"Checkpoint consistency check failed: {len(seed_features)} feature rows and "
        f"{len(seed_sequences)} sequence rows. Attach the complete Version 2 output."
    )
print("Seed feature rows:", len(seed_features))
print("Seed sequence rows:", len(seed_sequences))

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT)],
    check=True,
)
sys.path.insert(0, str(PROJECT / "src"))
importlib.invalidate_caches()


In [ ]:
# Shared stage recorder: a failed optional/network stage is visible but does not hide later outputs.
STAGE_LOG = []
STATE_FILE = DATA / "day6_run_state.json"

def run_stage(name, function):
    started = time.time()
    print(f"\n===== {name} =====", flush=True)
    try:
        result = function()
        row = {"stage": name, "status": "complete", "seconds": round(time.time() - started, 1)}
        if isinstance(result, dict):
            row["result"] = result
        print(json.dumps(row, indent=2, default=str), flush=True)
    except Exception as exc:
        row = {
            "stage": name,
            "status": "failed",
            "seconds": round(time.time() - started, 1),
            "error": f"{type(exc).__name__}: {exc}",
            "traceback": traceback.format_exc(limit=8),
        }
        print(row["error"], flush=True)
    STAGE_LOG.append(row)
    STATE_FILE.write_text(json.dumps(STAGE_LOG, indent=2, default=str), encoding="utf-8")
    return row

def get_inventory():
    manifest = pd.read_csv(DATA / "manifest.csv", dtype={"source_id": "string"})
    features = pd.read_csv(DATA / "features.csv", dtype={"source_id": "string"})
    usable_status = manifest["status"].astype(str).isin(["COMPLETE", "ARCHIVED"])
    tics = features["source_id"].astype("string")
    valid_tics = tics[tics.str.fullmatch(r"\d{8,16}", na=False)]
    return {
        "completed_or_archived_products": int(usable_status.sum()),
        "feature_observations": int(len(features)),
        "unique_tic_sources_with_features": int(valid_tics.nunique()),
        "feature_failures": int(pd.read_csv(DATA / "feature_errors.csv").shape[0])
        if (DATA / "feature_errors.csv").exists() else 0,
    }

In [ ]:
from cosmic_black_swan.fetch import fetch_tess
from cosmic_black_swan.features import featurize
from cosmic_black_swan.score import score_isolation_forest

TARGET_PRODUCTS = 20_000
SECTORS_PER_BATCH = 4       # At most 1,200 FITS files on disk before featurizing and pruning.
MAX_FETCH_PASSES = 2        # Retry sectors only if failed downloads leave the target short.
MIN_FREE_GB = 3.0
BATCH_LOG = WORK / "day6_batch_progress.json"


def archive_featurized_raw():
    """Delete only downloaded raw FITS already represented in features.csv."""
    feature_path = DATA / "features.csv"
    manifest_path = DATA / "manifest.csv"
    raw_root = (DATA / "raw").resolve()
    features = pd.read_csv(feature_path, usecols=["path"])
    processed_paths = set()
    for value in features["path"].dropna().astype(str):
        path = Path(value).resolve()
        if path.suffix.lower() == ".fits" and raw_root in path.parents:
            processed_paths.add(str(path))

    manifest = pd.read_csv(manifest_path, dtype={"source_id": "string"})
    if "local_path" not in manifest:
        return 0
    local_paths = manifest["local_path"].fillna("").astype(str)
    archived_indices = []
    deleted_files = 0
    for index, value in local_paths.items():
        if not value:
            continue
        path = Path(value).resolve()
        if path.suffix.lower() != ".fits" or raw_root not in path.parents:
            continue
        if str(path) not in processed_paths:
            continue
        if path.exists():
            path.unlink()
            deleted_files += 1
        archived_indices.append(index)

    if archived_indices:
        manifest.loc[archived_indices, "status"] = "ARCHIVED"
        manifest.loc[archived_indices, "local_path"] = ""
        if "message" in manifest:
            manifest["message"] = manifest["message"].astype("string")
            manifest.loc[archived_indices, "message"] = (
                "Featurized; raw FITS removed to stay within Kaggle disk quota."
            )
        manifest.to_csv(manifest_path, index=False)
    return deleted_files


def fetch_and_featurize_in_chunks():
    # Recover any already-downloaded files if this runtime retained partial work.
    before_recovery = get_inventory()
    featurize(DATA, append=True)
    recovered_deleted = archive_featurized_raw()
    print({
        "recovered_inventory": get_inventory(),
        "raw_fits_pruned": recovered_deleted,
        "inventory_before_recovery": before_recovery,
    }, flush=True)

    batch_records = []
    for pass_number in range(1, MAX_FETCH_PASSES + 1):
        pass_start = get_inventory()
        for first_sector in range(1, 97, SECTORS_PER_BATCH):
            current = get_inventory()
            if current["completed_or_archived_products"] >= TARGET_PRODUCTS:
                break
            free_gb = shutil.disk_usage(WORK).free / (1024 ** 3)
            if free_gb < MIN_FREE_GB:
                raise RuntimeError(
                    f"Stopping before disk fills: only {free_gb:.2f} GiB free. "
                    "Completed batches are preserved in the manifest/features."
                )

            sectors = list(range(first_sector, min(first_sector + SECTORS_PER_BATCH, 97)))
            print(
                f"Pass {pass_number}, sectors {sectors[0]}–{sectors[-1]}; "
                f"{current['completed_or_archived_products']}/{TARGET_PRODUCTS} products; "
                f"{free_gb:.1f} GiB free",
                flush=True,
            )
            fetch_result = fetch_tess(
                data_root=DATA,
                sectors=sectors,
                max_lightcurves=TARGET_PRODUCTS,
                per_sector_cap=300,
                seed=42,
                batch_size=32,
                pause_seconds=0.05,
            )
            feature_result = featurize(DATA, append=True)
            pruned = archive_featurized_raw()
            current = get_inventory()
            record = {
                "pass": pass_number,
                "sectors": sectors,
                "fetch": fetch_result,
                "feature": feature_result,
                "raw_fits_pruned": pruned,
                "inventory": current,
                "free_gib": round(shutil.disk_usage(WORK).free / (1024 ** 3), 2),
            }
            batch_records.append(record)
            BATCH_LOG.write_text(json.dumps(batch_records, indent=2, default=str), encoding="utf-8")
            print(json.dumps(record, indent=2, default=str), flush=True)

        pass_end = get_inventory()
        if pass_end["completed_or_archived_products"] >= TARGET_PRODUCTS:
            break
        if pass_end["completed_or_archived_products"] <= pass_start["completed_or_archived_products"]:
            print("No additional completed products this pass; stopping retries.", flush=True)
            break

    return {"inventory": get_inventory(), "batches": len(batch_records), "batch_log": str(BATCH_LOG)}


run_stage("fetch and featurize incrementally within disk quota", fetch_and_featurize_in_chunks)
run_stage(
    "refit Isolation Forest on all available feature rows",
    lambda: score_isolation_forest(DATA, contamination=0.02, seed=42),
)
print("Inventory after fetch/features/scoring:", json.dumps(get_inventory(), indent=2))

In [ ]:
from cosmic_black_swan.catalog_labels import build_catalog_labels
from cosmic_black_swan.classifier import train_xgboost
from cosmic_black_swan.conformal import run_conformal_screen
from cosmic_black_swan.injections import injection_recovery
from cosmic_black_swan.transformer import train_transformer

LABELS = DATA / "labels.csv"
run_stage("cross-match supported Gaia/TESS catalog labels", lambda: build_catalog_labels(DATA, match_radius_arcsec=2.0))

def fit_xgboost_if_supported():
    if not LABELS.exists():
        return {"status": "skipped", "reason": "No labels.csv was produced by the catalog cross-match."}
    labels = pd.read_csv(LABELS)
    if not {"source_id", "class_label"}.issubset(labels.columns):
        return {"status": "skipped", "reason": "Catalog labels lack source_id/class_label."}
    if labels["class_label"].dropna().nunique() < 2 or labels["source_id"].nunique() < 10:
        return {"status": "skipped", "reason": "Not enough supported catalog-labeled sources/classes for a held-out model."}
    return train_xgboost(DATA, LABELS, seed=42, minimum_evidence="single_catalog")

run_stage("source-disjoint XGBoost catalog-class baseline", fit_xgboost_if_supported)

def fit_transformer():
    if importlib.util.find_spec("torch") is None:
        return {"status": "skipped", "reason": "PyTorch is unavailable in this Kaggle runtime."}
    return train_transformer(DATA, epochs=5, batch_size=128, learning_rate=3e-4, seed=42)

run_stage("masked-patch Transformer held-out source scoring", fit_transformer)
run_stage("source-disjoint field-relative conformal ranking", lambda: run_conformal_screen(DATA, seed=42))
run_stage("held-out synthetic injection recovery and detector-location map", lambda: injection_recovery(DATA, sample_size=500, amplitudes_sigma=(1, 2, 4, 8), seed=42))

conformal_record = DATA / "scores/conformal_run.json"
if conformal_record.exists():
    conformal = json.loads(conformal_record.read_text(encoding="utf-8"))
    print("Conformal interpretation:", conformal.get("interpretation"))
    print("FDR control claimed by this implementation:", conformal.get("fdr_control_claim"))


In [ ]:
# Summarize what was actually produced; reaching the product count alone is not enough.
inventory = get_inventory()
score_dir = DATA / "scores"
isolation_run_path = score_dir / "run.json"
isolation_run = json.loads(isolation_run_path.read_text(encoding="utf-8")) if isolation_run_path.exists() else {}
scored_rows = int(isolation_run.get("input_rows", 0))
scored_target_reached = scored_rows >= TARGET_PRODUCTS
report = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "target_products": TARGET_PRODUCTS,
    "inventory": inventory,
    "isolation_forest_scored_rows": scored_rows,
    "target_scored_reached": scored_target_reached,
    "stages": STAGE_LOG,
    "artifact_paths": {
        name: str(path) for name, path in {
            "manifest": DATA / "manifest.csv",
            "features": DATA / "features.csv",
            "series": DATA / "series.npy",
            "isolation_forest_scores": score_dir / "isolation_forest.csv",
            "isolation_forest_run": isolation_run_path,
            "xgboost_run": score_dir / "xgboost_run.json",
            "transformer_run": score_dir / "transformer_run.json",
            "conformal_run": score_dir / "conformal_run.json",
            "injection_run": score_dir / "injection_recovery_run.json",
            "batch_progress": BATCH_LOG,
        }.items() if path.exists()
    },
    "claim_limits": {
        "raw_fits": "Downloaded light curves are deleted only after successful feature extraction; the manifest retains archive status and MAST identifiers. Re-download selected raw curves for detailed follow-up.",
        "fdr_control": "Not established: the current conformal reference is the observed field, not an independently vetted nominal population.",
        "catalog_unmatched": "Unmatched sources remain unlabeled; they are not treated as normal examples.",
        "score_semantics": "Anomaly scores and percentiles are screening ranks, not probabilities or discoveries.",
    },
}
report_path = WORK / "day6_run_summary.json"
report_path.write_text(json.dumps(report, indent=2, default=str), encoding="utf-8")
print("Saved:", report_path)
print(json.dumps({
    "inventory": inventory,
    "isolation_forest_scored_rows": scored_rows,
    "target_scored_reached": scored_target_reached,
    "stages": [{"stage": s["stage"], "status": s["status"]} for s in STAGE_LOG],
    "artifacts": report["artifact_paths"],
}, indent=2))